In [ ]:
import zipfile
import os

# Path to your ZIP file
zip_path = "/content/Dataset.zip"

# Folder where files will be extracted
extract_to = "/content"

# Create the output directory if it doesn't exist
os.makedirs(extract_to, exist_ok=True)

# Extract
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to)

print(f"Dataset extracted successfully to: {extract_to}")

Dataset extracted successfully to: /content


In [ ]:

# Install required packages

!pip install -q pymupdf
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q transformers
!pip install -q accelerate
!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-text-splitters
!pip install -q rank_bm25
!pip install -q networkx
!pip install -q pandas
!pip install -q numpy
!pip install -q tqdm
!pip install -q scikit-learn
!pip install -q google-generativeai
!pip install -q python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 82.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
# ============================================================
# IMPORTS
# ============================================================

import os
import re
import json
import fitz
import faiss
import pickle
import random
import warnings
import numpy as np
import pandas as pd
import networkx as nx

from pathlib import Path
from tqdm import tqdm

from sentence_transformers import SentenceTransformer
from sentence_transformers import CrossEncoder

from rank_bm25 import BM25Okapi

from langchain_text_splitters import RecursiveCharacterTextSplitter

import google.generativeai as genai

warnings.filterwarnings("ignore")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

PROJECT_PATH = "/content"

DATASET_PATH = os.path.join(PROJECT_PATH, "Dataset")

OUTPUT_PATH = os.path.join(PROJECT_PATH, "output")

os.makedirs(OUTPUT_PATH, exist_ok=True)

print(DATASET_PATH)

/content/Dataset


In [ ]:
# ============================================================
# RESUME FROM SAVED FILES
# ============================================================

import json
import pickle
import faiss
import numpy as np
import pandas as pd

# Load raw documents
with open(
    os.path.join(OUTPUT_PATH, "raw_documents.json"),
    "r"
) as f:
    documents = json.load(f)

# Load chunks
with open(
    os.path.join(OUTPUT_PATH, "chunks.json"),
    "r"
) as f:
    chunks = json.load(f)

# Load dataframe
chunk_df = pd.read_csv(
    os.path.join(OUTPUT_PATH, "chunks.csv")
)

# Load embeddings
embeddings = np.load(
    os.path.join(OUTPUT_PATH, "embeddings.npy")
)

# Load FAISS
index = faiss.read_index(
    os.path.join(OUTPUT_PATH, "faiss.index")
)

# Load BM25
with open(
    os.path.join(OUTPUT_PATH, "bm25.pkl"),
    "rb"
) as f:
    bm25 = pickle.load(f)

print("="*60)
print("PROJECT RESTORED SUCCESSFULLY")
print("="*60)
print(f"Documents : {len(documents)}")
print(f"Chunks    : {len(chunks)}")
print(f"Embeddings: {embeddings.shape}")
print(f"FAISS     : {index.ntotal}")

PROJECT RESTORED SUCCESSFULLY
Documents : 16133
Chunks    : 153975
Embeddings: (153975, 768)
FAISS     : 153975


In [ ]:
# ============================================================
# VERIFY DATASET
# ============================================================

categories = [
    "Acts",
    "Court_Judgements",
    "IRS",
    "POV"
]

for category in categories:

    folder = os.path.join(DATASET_PATH, category)

    pdfs = [f for f in os.listdir(folder) if f.endswith(".pdf")]

    print(f"{category:20s} : {len(pdfs)} PDFs")

Acts                 : 25 PDFs
Court_Judgements     : 24 PDFs
IRS                  : 25 PDFs
POV                  : 17 PDFs


In [ ]:
# ============================================================
# PDF PARSER
# ============================================================

def extract_pdf(pdf_path, category):

    document = fitz.open(pdf_path)

    pages = []

    for page_number in range(len(document)):

        page = document.load_page(page_number)

        text = page.get_text("text")

        text = re.sub(r"\s+", " ", text).strip()

        if len(text) < 50:
            continue

        pages.append({

            "document": os.path.basename(pdf_path),

            "category": category,

            "page": page_number + 1,

            "text": text

        })

    return pages

In [ ]:
# ============================================================
# READ COMPLETE DATASET
# ============================================================

documents = []

for category in categories:

    folder = os.path.join(DATASET_PATH, category)

    pdfs = sorted(os.listdir(folder))

    for pdf in tqdm(pdfs):

        if not pdf.endswith(".pdf"):
            continue

        pdf_path = os.path.join(folder, pdf)

        pages = extract_pdf(pdf_path, category)

        documents.extend(pages)

print()

print("Total Pages:", len(documents))

100%|██████████| 17/17 [00:04<00:00,  4.21it/s]


Total Pages: 16133


In [ ]:
# Display one sample

documents[0]

{'document': 'ACT1.pdf',
 'category': 'Acts',
 'page': 1,
 'text': 'Page 1 1 So in original. Does not conform to section catchline. 2 Section catchline amended by Pub. L. 117–228 without cor- responding amendment of chapter analysis. THE CODE OF LAWS OF THE UNITED STATES OF AMERICA TITLE 1—GENERAL PROVISIONS This title was enacted by act July 30, 1947, ch. 388, § 1, 61 Stat. 633 Chap. Sec. 1. Rules of construction ......................... 1 2. Acts and resolutions; formalities of enactment; repeals; sealing of in- struments ........................................... 101 3. Code of Laws of United States and Supplements; District of Colum- bia Code and Supplements ........... 201 Statutory Notes and Related Subsidiaries POSITIVE LAW; CITATION This title has been made positive law by section 1 of act July 30, 1947, ch. 388, 61 Stat. 633, which provided in part that: ‘‘Title 1 of the United States Code entitled ‘General Provisions’, is codified and enacted into posi- tive law and may be 

In [ ]:
# ============================================================
# SAVE RAW TEXT
# ============================================================

raw_path = os.path.join(
    OUTPUT_PATH,
    "raw_documents.json"
)

with open(raw_path, "w") as f:

    json.dump(documents, f, indent=2)

print("Saved:", raw_path)

Saved: /content/output/raw_documents.json


In [ ]:
# ============================================================
# TEXT SPLITTER
# ============================================================

text_splitter = RecursiveCharacterTextSplitter(

    chunk_size=800,

    chunk_overlap=150,

    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)

In [ ]:
# ============================================================
# CREATE CHUNKS
# ============================================================

chunks = []

chunk_id = 0

for page in tqdm(documents):

    split_text = text_splitter.split_text(page["text"])

    for piece in split_text:

        chunk = {

            "chunk_id": chunk_id,

            "document": page["document"],

            "category": page["category"],

            "page": page["page"],

            "text": piece

        }

        chunks.append(chunk)

        chunk_id += 1

print()

print("Total Chunks:", len(chunks))

100%|██████████| 16133/16133 [00:03<00:00, 4840.32it/s]


Total Chunks: 153975


In [ ]:
chunks[0]

{'chunk_id': 0,
 'document': 'ACT1.pdf',
 'category': 'Acts',
 'page': 1,
 'text': 'Page 1 1 So in original. Does not conform to section catchline. 2 Section catchline amended by Pub. L. 117–228 without cor- responding amendment of chapter analysis. THE CODE OF LAWS OF THE UNITED STATES OF AMERICA TITLE 1—GENERAL PROVISIONS This title was enacted by act July 30, 1947, ch. 388, § 1, 61 Stat. 633 Chap. Sec. 1. Rules of construction ......................... 1 2. Acts and resolutions; formalities of enactment; repeals; sealing of in- struments ........................................... 101 3. Code of Laws of United States and Supplements; District of Colum- bia Code and Supplements ........... 201 Statutory Notes and Related Subsidiaries POSITIVE LAW; CITATION This title has been made positive law by section 1 of act July 30, 1947, ch. 388, 61 Stat'}

In [ ]:
# ============================================================
# CHUNK STATISTICS
# ============================================================

lengths = [len(c["text"]) for c in chunks]

print("Chunks:", len(chunks))

print("Average Length:", np.mean(lengths))

print("Minimum:", np.min(lengths))

print("Maximum:", np.max(lengths))

Chunks: 153975
Average Length: 643.07846728365
Minimum: 3
Maximum: 800


In [ ]:
# ============================================================
# SAVE CHUNKS
# ============================================================

chunk_path = os.path.join(

    OUTPUT_PATH,

    "chunks.json"
)

with open(chunk_path, "w") as f:

    json.dump(chunks, f, indent=2)

print("Saved:", chunk_path)

Saved: /content/output/chunks.json


In [ ]:
# ============================================================
# DATAFRAME
# ============================================================

chunk_df = pd.DataFrame(chunks)

chunk_df.head()

,chunk_id,document,category,page,text
0,0,ACT1.pdf,Acts,1,Page 1 1 So in original. Does not conform to s...
1,1,ACT1.pdf,Acts,1,". 388, 61 Stat. 633, which provided in part th..."
2,2,ACT1.pdf,Acts,1,". WRITS OF ERROR Act June 25, 1948, ch. 646, §..."
3,3,ACT1.pdf,Acts,1,". 4 5 ............ R.S., § 5 ...................."
4,4,ACT1.pdf,Acts,1,". 106 Mar. 2, 1895, ch. 177, § 1, 28 Stat. 769..."


In [ ]:
chunk_df.to_csv(

    os.path.join(

        OUTPUT_PATH,

        "chunks.csv"

    ),

    index=False
)

In [ ]:
chunk_df.groupby("category").size()

,0
category,
Acts,132160
Court_Judgements,4521
IRS,10731
POV,6563


In [ ]:
# ============================================================
# LOAD EMBEDDING MODEL
# ============================================================

EMBEDDING_MODEL = "BAAI/bge-base-en-v1.5"

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL,
    device="cuda"
)

print("Embedding Model Loaded")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Model Loaded


In [ ]:
# ============================================================
# PREPARE TEXTS
# ============================================================

texts = [chunk["text"] for chunk in chunks]

print("Total Chunks :", len(texts))

Total Chunks : 153975


In [ ]:
# ============================================================
# GENERATE EMBEDDINGS
# ============================================================

embeddings = embedding_model.encode(

    texts,

    batch_size=480,

    show_progress_bar=True,

    convert_to_numpy=True,

    normalize_embeddings=True
)

print(embeddings.shape)

Batches:   0%|          | 0/321 [00:00<?, ?it/s]

(153975, 768)


In [ ]:
# ============================================================
# SAVE EMBEDDINGS
# ============================================================

embedding_path = os.path.join(
    OUTPUT_PATH,
    "embeddings.npy"
)

np.save(
    embedding_path,
    embeddings
)

print("Saved")

Saved


In [ ]:
# ============================================================
# BUILD FAISS INDEX
# ============================================================

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print(index.ntotal)

153975


In [ ]:
# ============================================================
# SAVE INDEX
# ============================================================

faiss.write_index(

    index,

    os.path.join(
        OUTPUT_PATH,
        "faiss.index"
    )
)

In [ ]:
import re

def tokenize(text):
    """
    Tokenize text by extracting alphanumeric words.
    This is more robust than text.split() for legal documents.
    """
    return re.findall(r"\b[a-zA-Z0-9]+\b", text.lower())

In [ ]:
# ============================================================
# TOKENIZE
# ============================================================

# ============================================================
# TOKENIZATION FOR BM25
# ============================================================

import re

def tokenize(text):
    """
    Tokenize text by extracting alphanumeric words.
    This is more robust than text.split() for legal documents.
    """
    return re.findall(r"\b[a-zA-Z0-9]+\b", text.lower())

# Tokenize every chunk
tokenized_chunks = [tokenize(chunk["text"]) for chunk in chunks]

# Build BM25 index
bm25 = BM25Okapi(tokenized_chunks)

print("BM25 Index Ready!")

BM25 Index Ready!


In [ ]:
# ============================================================
# SAVE BM25
# ============================================================

with open(

    os.path.join(
        OUTPUT_PATH,
        "bm25.pkl"
    ),

    "wb"

) as f:

    pickle.dump(bm25, f)

In [ ]:
# ============================================================
# VECTOR SEARCH
# ============================================================

def vector_search(query, top_k=5):

    query_embedding = embedding_model.encode(

        [query],

        normalize_embeddings=True,

        convert_to_numpy=True

    )

    scores, indices = index.search(

        query_embedding,

        top_k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):

        item = chunks[idx].copy()

        item["score"] = float(score)

        results.append(item)

    return results

In [ ]:
results = vector_search(

    "Can business expenses be deducted?"

)

for r in results:

    print("="*80)

    print("Score :", round(r["score"],3))

    print("Document :", r["document"])

    print("Page :", r["page"])

    print()

    print(r["text"][:400])

Score : 0.814
Document : p7.pdf
Page : 32

8. Business Expenses Introduction You can deduct the costs of operating your business. These costs are known as business expenses. These are costs you don’t have to capitalize or include in the cost of goods sold but can deduct in the current year. To be deductible, a business expense must be both or- dinary and necessary. An ordinary expense is one that is common and accepted in your field of bus
Score : 0.786
Document : p7.pdf
Page : 41

. See Pub. 463 for additional information. Cleaning. You can deduct the costs of dry cleaning and laundry while on your business trip. Telephone. You can deduct the cost of business calls while on your business trip, including business communi- cation by fax machine or other communication devices. Tips. You can deduct the tips you pay for any expense in this list. More information. For more inform
Score : 0.785
Document : p7.pdf
Page : 42

ways to qualify to deduct home office expenses, see Pub. 587. Deducti

In [ ]:
# ============================================================
# BM25 SEARCH
# ============================================================

def bm25_search(query, top_k=5):

    tokens = tokenize(query)

    scores = bm25.get_scores(tokens)

    ranked = np.argsort(scores)[::-1][:top_k]

    results = []

    for idx in ranked:

        item = chunks[idx].copy()

        item["score"] = float(scores[idx])

        results.append(item)

    return results

In [ ]:
results = bm25_search(

    "Section 179 deduction"

)

for r in results:

    print("="*80)

    print(r["document"])

    print(r["page"])

    print(r["score"])

p8.pdf
22
20.243922953089545
p5.pdf
43
19.73304455799532
p8.pdf
23
19.5999292713272
p5.pdf
43
19.143771132306217
p8.pdf
24
18.979488184848513


In [ ]:
# ============================================================
# RECIPROCAL RANK FUSION
# ============================================================

def reciprocal_rank_fusion(vector_results,
                            bm25_results,
                            k=60):

    fused_scores = {}

    document_map = {}

    # ------------------------
    # Vector Search
    # ------------------------

    for rank, item in enumerate(vector_results):

        cid = item["chunk_id"]

        document_map[cid] = item

        fused_scores[cid] = fused_scores.get(cid, 0) + 1 / (k + rank + 1)

    # ------------------------
    # BM25 Search
    # ------------------------

    for rank, item in enumerate(bm25_results):

        cid = item["chunk_id"]

        document_map[cid] = item

        fused_scores[cid] = fused_scores.get(cid, 0) + 1 / (k + rank + 1)

    ranked = sorted(

        fused_scores.items(),

        key=lambda x: x[1],

        reverse=True

    )

    final_results = []

    for cid, score in ranked:

        item = document_map[cid].copy()

        item["hybrid_score"] = score

        final_results.append(item)

    return final_results

In [ ]:
# ============================================================
# HYBRID SEARCH
# ============================================================

def hybrid_search(

        query,

        vector_k=15,

        bm25_k=15,

        final_k=10

):

    vector_results = vector_search(

        query,

        top_k=vector_k

    )

    bm25_results = bm25_search(

        query,

        top_k=bm25_k

    )

    fused = reciprocal_rank_fusion(

        vector_results,

        bm25_results

    )

    return fused[:final_k]

In [ ]:
results = hybrid_search(

    "Can business expenses be deducted?"

)

for r in results:

    print("="*80)

    print("Document :", r["document"])

    print("Category :", r["category"])

    print("Page :", r["page"])

    print("Hybrid Score :", round(r["hybrid_score"],4))

    print()

    print(r["text"][:300])

    print()

Document : p18.pdf
Category : IRS
Page : 7
Hybrid Score : 0.03

Generally, entertainment and entertainment-related meal expenses, membership dues, and facilities used in con- nection with these activities cannot be deducted. Primary purpose of trip must be for business. If your trip was entirely for business, your unreimbursed travel ex- penses are generally ded

Document : p7.pdf
Category : IRS
Page : 32
Hybrid Score : 0.0164

8. Business Expenses Introduction You can deduct the costs of operating your business. These costs are known as business expenses. These are costs you don’t have to capitalize or include in the cost of goods sold but can deduct in the current year. To be deductible, a business expense must be both o

Document : p24.pdf
Category : IRS
Page : 12
Hybrid Score : 0.0164

. Example. You are blind. You must use a reader to do your work. You use the reader both during your regular working hours at your place of work and outside your reg- ular working hours away from you

In [ ]:
# ============================================================
# LOAD CROSS ENCODER
# ============================================================

RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

reranker = CrossEncoder(
    RERANK_MODEL,
    device="cuda"
)

print("Cross Encoder Loaded")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Cross Encoder Loaded


In [ ]:
def diversify_results(results):

    unique = []

    seen = set()

    for item in results:

        key = (
            item["document"],
            item["page"]
        )

        if key in seen:
            continue

        seen.add(key)

        unique.append(item)

    return unique

In [ ]:
def rerank_results(
        query,
        retrieved_chunks,
        top_k=5
):

    pairs = [

        (query, chunk["text"])

        for chunk in retrieved_chunks

    ]

    scores = reranker.predict(pairs)

    ranked = sorted(

        zip(scores, retrieved_chunks),

        key=lambda x: x[0],

        reverse=True

    )

    final = []

    for score, chunk in ranked[:top_k]:

        item = chunk.copy()

        item["rerank_score"] = float(score)

        final.append(item)

    return final

In [ ]:
# ============================================================
# COMPLETE RETRIEVAL PIPELINE
# ============================================================

def retrieve_context(

        query,

        top_k=5

):

    # Step 1
    hybrid = hybrid_search(

        query,

        vector_k=20,

        bm25_k=20,

        final_k=20

    )

    # Step 2
    hybrid = diversify_results(hybrid)

    # Step 3
    reranked = rerank_results(

        query,

        hybrid,

        top_k=top_k

    )

    return reranked

In [ ]:
results = retrieve_context(

    "Can business expenses be deducted?"

)

for r in results:

    print("="*80)

    print("Document :", r["document"])

    print("Page :", r["page"])

    print("Category :", r["category"])

    print("Cross Score :", round(r["rerank_score"],3))

    print()

    print(r["text"][:400])

    print()

Document : p7.pdf
Page : 32
Category : IRS
Cross Score : 7.947

8. Business Expenses Introduction You can deduct the costs of operating your business. These costs are known as business expenses. These are costs you don’t have to capitalize or include in the cost of goods sold but can deduct in the current year. To be deductible, a business expense must be both or- dinary and necessary. An ordinary expense is one that is common and accepted in your field of bus

Document : p7.pdf
Page : 42
Category : IRS
Cross Score : 7.606

ways to qualify to deduct home office expenses, see Pub. 587. Deduction limit. If your gross income from the business use of your home equals or exceeds your total business expenses (including depreciation), you can deduct all your business expenses related to the use of your home. If your gross income from the business use is less than your total business expenses, your deduction for certain expe

Document : p2.pdf
Page : 103
Category : IRS
Cross Score : 6.986

. U

In [ ]:
# ============================================================
# REMOVE DUPLICATE CHUNKS
# ============================================================

def diversify_results(results):

    unique = []

    seen = set()

    for item in results:

        key = (
            item["document"],
            item["page"]
        )

        if key in seen:
            continue

        seen.add(key)

        unique.append(item)

    return unique

In [ ]:
# ============================================================
# GEMINI CONFIGURATION
# ============================================================

import google.generativeai as genai

GOOGLE_API_KEY = "KEY"

genai.configure(api_key=GOOGLE_API_KEY)

In [ ]:
# ============================================================
# LOAD GEMINI MODEL
# ============================================================

model = genai.GenerativeModel(
    "gemini-3.5-flash"
)

print("Gemini Loaded")

Gemini Loaded


In [ ]:
# ============================================================
# BUILD CONTEXT
# ============================================================

def build_context(retrieved_chunks):

    context = ""

    for i, chunk in enumerate(retrieved_chunks, start=1):

        context += f"""
==============================
Context {i}

Document : {chunk['document']}
Category : {chunk['category']}
Page : {chunk['page']}

Content:
{chunk['text']}

"""

    return context

In [ ]:
# ============================================================
# PROMPT TEMPLATE
# ============================================================

SYSTEM_PROMPT = """
You are an expert US Tax and Legal assistant.

Rules:

1. Answer ONLY using the provided context.

2. Never invent legal information.

3. If the answer cannot be found in the retrieved context,
reply exactly:

'I could not find sufficient information in the provided legal documents.'

4. Every factual statement must be supported
by the retrieved documents.

5. At the end include citations.

Citation format:

Document:
Page:

6. Keep the answer concise and professional.
"""

In [ ]:
# ============================================================
# ASK GEMINI
# ============================================================

def ask_gemini(question, context):

    prompt = f"""

{SYSTEM_PROMPT}

Question:

{question}

Legal Context:

{context}

Answer:

"""

    response = model.generate_content(prompt)

    return response.text

In [ ]:
def legal_rag(question):

    retrieved = retrieve_context(
        question,
        top_k=5
    )

    context = build_context(retrieved)

    answer = ask_gemini(
        question,
        context
    )

    citations = []

    seen = set()

    for chunk in retrieved:

        key = (
            chunk["document"],
            chunk["page"]
        )

        if key not in seen:

            seen.add(key)

            citations.append(
                f"{chunk['document']} (Page {chunk['page']})"
            )

    return {
        "question": question,
        "answer": answer,
        "citations": citations,
        "sources": retrieved
    }

In [ ]:
result = legal_rag(
    "Can business meals be deducted?"
)

print("="*80)
print("ANSWER")
print("="*80)

print(result["answer"])

print()

print("="*80)
print("REFERENCES")
print("="*80)

for citation in result["citations"]:
    print("•", citation)

ERROR:tornado.access:503 POST /v1beta/models/gemini-3.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2792.12ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-3.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 5110.09ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-3.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2189.22ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-3.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1964.69ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-3.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2238.51ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-3.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 12827.09ms


TooManyRequests: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-3.5-flash
Please retry in 10.344559159s.

In [ ]:
# ============================================================
# GET FULL DOCUMENT
# ============================================================

def get_document_text(document_name):

    doc_chunks = [

        chunk

        for chunk in chunks

        if chunk["document"] == document_name

    ]

    doc_chunks = sorted(

        doc_chunks,

        key=lambda x: x["page"]

    )

    text = ""

    for chunk in doc_chunks:

        text += chunk["text"] + "\n\n"

    return text

In [ ]:
# ============================================================
# LEGAL SUMMARY PROMPT
# ============================================================

SUMMARY_PROMPT = """
You are an expert legal assistant.

Summarize the following legal document.

Requirements:

- Keep the summary concise.

- Mention important legal principles.

- Mention important tax provisions.

- Mention major conclusions.

- Do not invent information.

Return the summary in professional language.
"""

In [ ]:
# ============================================================
# DOCUMENT SUMMARIZER
# ============================================================
import numpy as np
def summarize_document(document_name, max_chunks=100):

    doc_chunks = sorted(
        [c for c in chunks if c["document"] == document_name],
        key=lambda x: x["page"]
    )

    indices = np.linspace(
                  0,
                  len(doc_chunks) - 1,
                  max_chunks,
                  dtype=int
    )

    selected_chunks = [doc_chunks[i] for i in indices]

    text = "\n\n".join(
      chunk["text"]
      for chunk in selected_chunks
    )

    prompt = f"""
{SUMMARY_PROMPT}

Document:
{document_name}

Content:
{text}
"""

    print(f"Chunks used: {len(doc_chunks[:max_chunks])}")
    print(f"Text length: {len(text)} characters")
    print(f"Prompt length: {len(prompt)} characters")

    response = model.generate_content(prompt)

    return response.text

In [ ]:
doc_chunks = [c for c in chunks if c["document"] == "ACT3.pdf"]

print("Number of chunks:", len(doc_chunks))

total_chars = sum(len(c["text"]) for c in doc_chunks)
print("Characters:", total_chars)

Number of chunks: 789
Characters: 504138


In [ ]:
import time

start = time.time()

response = model.generate_content("Hello")

print(response.text)

print(f"Time: {time.time() - start:.2f} seconds")

import time

start = time.time()

response = model.generate_content(
    "Summarize this in one sentence: "
    + ("Artificial intelligence is transforming legal research. " * 50)
)

print(response.text)

print(f"Time: {time.time() - start:.2f} seconds")

KeyboardInterrupt: 

In [ ]:
summary = summarize_document(

    "ACT3.pdf",
    max_chunks=20

)

print(summary)

Chunks used: 20
Text length: 11651 characters
Prompt length: 11987 characters


KeyboardInterrupt: 